In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas11-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [18]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [15]:
#Feature Engineering
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

idx_menu_model   = StringIndexer(inputCol="nama_menu",   outputCol="menu_idx").fit(df)
idx_cabang_model = StringIndexer(inputCol="cabang",      outputCol="cabang_idx").fit(df)
idx_waktu_model  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df)

df_idx = idx_menu_model.transform(df)
df_idx = idx_cabang_model.transform(df_idx)
df_idx = idx_waktu_model.transform(df_idx)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_idx)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [22]:
#Train/Test
from pyspark.ml.regression import LinearRegression
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

# Pelatihan model
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Data training: 1839 | Data testing: 661
Koefisien: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


In [25]:
#Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator
predictions = model.transform(test)

eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2   = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = eval_rmse.evaluate(predictions)
r2   = eval_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


NIlai diatas cukup baik untuk estimasi omzet kasar dan perencanaan stok. Namun model masih memiliki kekurangan dalam memprediksi rata rata per pesanan, sehingga belum ideal untuk prediksi yang sangat presisi.

In [26]:
#Eksperimen Perbandingan Fitur
assembler_dasar = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_dasar")

train_dasar = assembler_dasar.transform(train)
test_dasar  = assembler_dasar.transform(test)

lr_dasar = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_dasar = lr_dasar.fit(train_dasar)
pred_dasar = model_dasar.transform(test_dasar)

r2_dasar = eval_r2.evaluate(pred_dasar)

print(f"R² model utama (5 fitur)      : {r2:.4f}")
print(f"R² model dasar (qty + harga)  : {r2_dasar:.4f}")
print(f"Selisih                       : {r2 - r2_dasar:.4f}")

R² model utama (5 fitur)      : 0.9107
R² model dasar (qty + harga)  : 0.9108
Selisih                       : -0.0001


In [27]:
#Prediksi Skenario Baru
from pyspark.sql.functions import abs as abs_, col
skenario = spark.createDataFrame([
    ("Kopi Susu",  "Magelang",   "Pagi",  3, 18000),
    ("Cappuccino", "Yogyakarta", "Sore",  2, 25000),
    ("Roti Bakar", "Semarang",   "Malam", 5, 20000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

skenario_idx = idx_menu_model.transform(skenario)
skenario_idx = idx_cabang_model.transform(skenario_idx)
skenario_idx = idx_waktu_model.transform(skenario_idx)

skenario_final = assembler.transform(skenario_idx)

hasil = model.transform(skenario_final)
hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |3  |18000       |53526.17351912394 |
|Cappuccino|Yogyakarta|Sore       |2  |25000       |53814.94297982256 |
|Roti Bakar|Semarang  |Malam      |5  |20000       |104193.90687409374|
+----------+----------+-----------+---+------------+------------------+

